Гуминов Дмитрий Андреевич ИИ25

### Домашнее задание 2 (часть 1): дообучение BERT

В этом задании вы получите эмбеддинги с помощью BERT, дообучите предобученную модель на новых данных и сравните возможности трансформеров с предыдущими архитектурами, учитывая их более высокие вычислительные затраты.

**Всего: 10 баллов (2 + 5 + 3); все части обязательны.** Сдайте только заполненный ноутбук с сохранёнными результатами оценки и сравнения моделей. Баллы за код второй части линейно масштабируются по accuracy на полной валидационной выборке: 0 при 90% и ниже, все заработанные баллы при 92% и выше. Веса моделей и файлы эмбеддингов сдавать не нужно.

Заполните четыре ячейки TODO, выполните проверки и сохраните результаты оценки и сравнения в сдаваемом ноутбуке.


In [1]:
# %pip install --quiet "transformers==4.57.6" "datasets==4.5.0" "accelerate==1.12.0" "sentencepiece==0.2.1" "protobuf==5.29.5" "scikit-learn>=1.5,<2"

import os
os.environ["WANDB_DISABLED"] = "true"

import torch
import transformers
import datasets
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, Trainer, TrainingArguments


### Загрузка данных и модели


В этом задании используется **Quora Question Pairs (QQP)**.

Датасет содержит более 400 000 пар вопросов. Для каждой пары указано, являются ли вопросы перефразировками друг друга: речь идёт о дубликатах по смыслу, а не просто о связанных темах. Подробнее о данных можно прочитать [здесь](https://paperswithcode.com/dataset/quora-question-pairs).
Метки: 0 — не дубликаты, 1 — дубликаты.


In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Пример [0]:", qqp['train'][0])
print("Пример [3]:", qqp['train'][3])


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Пример [0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Пример [3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
# Открытые проверки
for split in ['train', 'validation']:
    assert len(qqp[split]) > 0, f'{split}: датасет пуст.'
    assert set(qqp[split]['label']) == {0, 1}, f'{split}: ожидаются метки классов 0 и 1.'
    assert {'text1', 'text2', 'label'}.issubset(qqp[split].column_names), 'Сохраните оба вопроса и их метку.'


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Токенизация данных


Библиотека [datasets](https://huggingface.co/docs/datasets/en/index) позволяет применять функцию к данным с помощью `map`, как в функциональном программировании.

Что происходит с текстами в `qqp_preprocessed`?

- Исходные `text1` и `text2` преобразуются в числовые идентификаторы токенов с помощью соответствующего токенизатора.
- Для BERT тексты объединяются с разделителем `SEP`, а в начало добавляется токен `CLS`. Последовательность обрезается, если её длина превышает 128 токенов, или дополняется до 128 токенов, если она короче.
- Датасет `qqp_preprocessed` содержит:
    - _Идентификаторы токенов_ (`input_ids`): последовательность числовых идентификаторов.
    - _Маски внимания_ (`attention_mask`): бинарные маски, обозначающие позиции дополнения.
    - _Идентификаторы типов токенов_ (`token_type_ids`): различают токены из `text1` и `text2`.

__Обратите внимание!__ Маски внимания исключают позиции дополнения из внимания, но не отменяют все вычисления на этих позициях. `token_type_ids` могут отсутствовать.


In [5]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [6]:
# Открытые проверки
assert model.config.num_labels == 2, 'Ожидаются два логита классов.'
for split in ['train', 'validation']:
    assert len(qqp_preprocessed[split]) == len(qqp[split]), 'Токенизация должна сохранять число строк.'
    np.testing.assert_array_equal(qqp_preprocessed[split]['label'], qqp[split]['label'], err_msg='Сохраните метки и порядок строк.')
    sample = qqp_preprocessed[split][0]
    assert len(sample['input_ids']) == len(sample['attention_mask']) == MAX_LENGTH, 'Дополните или обрежьте последовательность до MAX_LENGTH.'
    assert set(sample['attention_mask']).issubset({0, 1}), 'Маска внимания должна содержать только 0 и 1.'
    assert 0 <= min(sample['input_ids']) <= max(sample['input_ids']) < model.config.vocab_size, 'Идентификаторы токенов должны принадлежать словарю этой модели.'


In [7]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")


[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Оценка качества (2 балла)

Мы выбрали готовую модель, обученную на QQP. Насколько хорошо она работает?

Один из способов это проверить — вычислить accuracy на валидационной выборке. Далее вы реализуете эту оценку.

Ниже приведён код, с которого можно начать:


Посмотрим на наши данные.


In [8]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)


In [9]:
for batch in val_loader:
     break  # Посмотрим на один батч.
print("Пример батча:", batch)

model.eval()
with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nПредсказание (вероятности):', torch.softmax(predicted.logits, dim=1).data.numpy())


Пример батча: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

У модели одна классификационная голова с двумя выходными логитами — по одному на каждый класс.


__Ваша задача__ — вычислить accuracy модели на валидационной выборке.
Наивная реализация может работать несколько часов. Используйте следующие оптимизации:

- запускайте модель на GPU с `no_grad`;
- используйте размер батча больше 1;
- начните с `num_workers=0`; при необходимости измерьте скорость с дополнительными процессами загрузки данных;
- при желании используйте [смешанную точность](https://pytorch.org/docs/stable/notes/amp_examples.html).


Хотя вычисления модели выполняются на GPU, загрузка данных с диска или из памяти и преобразование в нужный модели формат, например в тензоры, выполняются на CPU.

Недостаточная производительность CPU может замедлять весь процесс.


In [10]:
from tqdm import tqdm
import multiprocessing

cores = multiprocessing.cpu_count() # Число ядер процессора.
cores


16

In [ ]:
# Переносим модель на GPU, если он доступен.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Создаём DataLoader для валидационной выборки.
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16,  # Увеличенный батч ускоряет обработку.
    shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=4 # 0  # Подходит для Colab; дополнительные процессы загрузки данных — необязательная оптимизация.
)


In [20]:
# Оценка accuracy на валидации.
# TODO 1: замените каждое ... кодом для обработки всех батчей и вычисления accuracy.
model.eval()
correct_predictions = 0
total_predictions = 0
use_amp = device.type == 'cuda'

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Оценка качества"):
        # Перенесите входы модели и метки на устройство; исключите служебные поля строк.
        inputs = {
            'input_ids': batch['input_ids'].to(device),
            'attention_mask': batch['attention_mask'].to(device),
            'token_type_ids': batch['token_type_ids'].to(device),
        } # раскладываю как нужно модели 
        labels = batch['labels'].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(**inputs)
        # Получите предсказания классов и подсчитайте примеры, включая последний батч.
        preds = outputs.logits.argmax(dim=1)
        correct_predictions += (preds == labels).sum().item()
        total_predictions += len(labels)

accuracy = correct_predictions / total_predictions  # Доля правильных ответов от 0 до 1.
print(f"Accuracy на валидации: {accuracy:.4f}")


Оценка качества: 100%|██████████| 2527/2527 [00:30<00:00, 82.79it/s] 

Accuracy на валидации: 0.9083


In [21]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy)
assert 0 <= accuracy <= 1
assert total_predictions == len(val_set), 'Обработайте каждый пример валидации ровно один раз.'
assert np.isclose(accuracy, correct_predictions / total_predictions)
baseline_accuracy = float(accuracy)
print('Опубликованная accuracy базовой модели на полной выборке — около 0.9084; другая длина или настройки могут изменить результат.')


Опубликованная accuracy базовой модели на полной выборке — около 0.9084; другая длина или настройки могут изменить результат.


### Дообучение модели (5 баллов)

В этой части вам нужно самостоятельно дообучить модель. Выберите предобученную модель __с архитектурой, отличной от оригинального BERT__, которую ранее не дообучали на QQP, в том числе в рамках GLUE или обучения на нескольких задачах. Рекомендуем [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Можно выбрать другую модель, опираясь на результаты публичных бенчмарков, например [GLUE](https://gluebenchmark.com/).

Вы можете написать цикл обучения самостоятельно или использовать `transformers.Trainer` (см. [пример](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Стремитесь к accuracy 92% на полной валидационной выборке. Оценка за вторую часть равна `C * clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1)`, где `C` — баллы за корректность кода, от 0 до 5. При accuracy 90% и ниже оценка равна 0; при 92% и выше вы получаете все заработанные за код баллы. Сравните результат с готовой моделью BERT из первой части. Оценивайте обе модели в одинаковых условиях, обучайтесь только на тренировочной выборке и сохраните вывод сравнения. Если вы остановили обучение раньше, оцените текущую модель.


In [28]:
# TODO 2: выберите модель с архитектурой не BERT, ранее не дообученную на QQP, и токенизируйте пары.
# Освобождаем память GPU, занятую базовой моделью, перед загрузкой новой.
model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Загрузите выбранную модель и её токенизатор, например DeBERTa-v3.
model_name = "microsoft/deberta-v3-base" # решил не экспериментировать с моделями :)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, dtype=torch.float32)  # Загрузите с num_labels=2 и dtype=torch.float32 для обучения с AMP.

# Повторно токенизируйте qqp новым токенизатором; можно использовать функцию предобработки выше.

# Определите функцию токенизации пар или используйте уже написанную.


# Токенизируйте qqp, сохранив метки и порядок строк в выборках.
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

/home/uuu10/projects/hse/nlp-1-2026/.venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

In [29]:
# Открытые проверки
assert model.config.num_labels == 2, 'Используйте два логита для классификации на два класса.'
assert model.config.model_type != 'bert', 'Выберите архитектуру, отличную от оригинального BERT.'
for split in ['train', 'validation']:
    raw, encoded = qqp[split], qqp_preprocessed[split]
    assert len(encoded) == len(raw), 'Сохраните исходный размер выборки.'
    np.testing.assert_array_equal(encoded['label'], raw['label'], err_msg='Сохраните метки и порядок строк.')
    probe_ids = [0, len(raw) // 2, len(raw) - 1]
    raw_probe = raw.select(probe_ids)
    expected = tokenizer(list(raw_probe['text1']), list(raw_probe['text2']), padding='max_length',
                         truncation=True, max_length=MAX_LENGTH)
    np.testing.assert_array_equal(encoded.select(probe_ids)['input_ids'], expected['input_ids'],
                                  err_msg='Повторно токенизируйте оба исходных вопроса новым токенизатором.')
accuracy = None  # После прерванного обучения не используйте прежнюю accuracy модели BERT.


In [30]:
# TODO 3: реализуйте метрику, настройте и запустите обучение, оцените качество.
# Подготавливаем тренировочную и валидационную выборки.
train_set = qqp_preprocessed['train']
val_set = qqp_preprocessed['validation']  


# Задаём метрику оценки качества. При желании напишите собственную реализацию.
from sklearn.metrics import accuracy_score

# При использовании transformers.Trainer можно воспользоваться функцией ниже.
def compute_metrics(eval_pred):
    """
    Вычисляет метрики при обучении или оценке модели.
    Аргументы:
        eval_pred (tuple): Кортеж, содержащий:
            - logits (ndarray или torch.Tensor): Логиты модели для каждого примера
              в батче оценки. Форма: (batch_size, num_classes).
            - labels (ndarray или torch.Tensor): Истинные метки каждого примера.
              Форма: (batch_size,).
    Возвращает:
        dict: Словарь вычисленных метрик:
            - "accuracy" (float): Доля правильных ответов среди всех примеров.
    """
    preds = eval_pred[0].argmax(axis=1)
    accuracy = (preds == eval_pred[1]).sum().item() / len(preds)  # Вычислите по eval_pred, не используя глобальную переменную accuracy.
    return {"accuracy": accuracy}

# Открытые проверки
metric_logits = np.array([[4, 1], [1, 4], [4, 1], [1, 4], [1, 4]])
metric_labels = np.array([0, 1, 0, 1, 0])
assert np.isclose(compute_metrics((metric_logits, metric_labels))['accuracy'], 0.8), 'Вычислите accuracy по примерам, выбрав предсказания вдоль измерения классов.'


In [31]:
# При желании напишите цикл обучения самостоятельно вместо transformers.Trainer.
# Начальное значение скорости обучения — 2e-5.
# При необходимости меняйте скорость обучения на порядок, например до 2e-4 или 2e-3.
# Трёх эпох обычно достаточно для дообучения без сильного забывания ранее изученного.
# Используйте регуляризацию weight_decay. Начните с 1e-2 и при необходимости измените значение.
# Можно выбирать лучшую модель по accuracy.

# Задайте параметры обучения без аргумента device: устройство выбирается автоматически.
# Оценивайте и сохраняйте модель каждые 1000 шагов оптимизатора; записывайте loss каждые 50 шагов.
training_args = TrainingArguments(
    output_dir="./deberta-qqp",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=1e-2,

    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",

    # добавлю накопление градиентов
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
)

# Оставляем веса в FP32 для AMP и очищаем градиенты от предыдущего неудачного запуска.
model.float()
model.zero_grad(set_to_none=True)

# Классификатор возвращает средний loss батча; Trainer учитывает накопление градиентов.
model.accepts_loss_kwargs = False

# Создаём Trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=val_set,
    compute_metrics=compute_metrics,
    data_collator=transformers.default_data_collator,
)

# Дообучаем модель.
trainer.train()

# Оцените текущую модель на валидации и запишите результат в accuracy.
metrics = trainer.evaluate()
accuracy = metrics['eval_accuracy']

print(f"Accuracy на валидации: {accuracy:.4f}")

Step,Training Loss,Validation Loss,Accuracy
1000,0.276100,0.279726,0.876997
2000,0.257000,0.282989,0.876849
3000,0.260200,0.255392,0.893816
4000,0.256200,0.245324,0.895770
5000,0.244800,0.245830,0.899580
6000,0.212600,0.239296,0.903463
7000,0.216800,0.226833,0.906777
8000,0.268500,0.234375,0.902721
9000,0.215200,0.224251,0.910685
10000,0.236600,0.218780,0.909547


Accuracy на валидации: 0.9244


In [32]:
assert isinstance(accuracy, (float, int, np.floating)) and np.isfinite(accuracy), 'Оцените дообученную модель; не используйте accuracy базовой модели.'
assert 0 <= accuracy <= 1
if 'trainer' in globals():
    assert trainer.state.global_step > 0, 'Запустите обучение перед итоговой оценкой.'
    logged_losses = [record[key] for record in trainer.state.log_history for key in ('loss', 'train_loss') if key in record]
    assert all(np.isfinite(value) for value in logged_losses), 'Значения функции потерь при обучении должны быть конечными.'
import json
comparison = {
    'baseline_accuracy': float(baseline_accuracy),
    'finetuned_accuracy': float(accuracy),
    'delta': float(accuracy - baseline_accuracy),
    'validation_examples': len(qqp['validation']),
}
print(f'Базовая модель BERT: {baseline_accuracy:.6f}; ваша модель: {accuracy:.6f}; прирост: {accuracy - baseline_accuracy:+.6f}')
print('Результат базовой модели BERT превышен:', accuracy > baseline_accuracy)
accuracy_factor = float(np.clip((accuracy - 0.90) / (0.92 - 0.90), 0, 1))
print(f'Коэффициент качества второй части: {accuracy_factor:.4f}; максимум баллов за вторую часть при этой accuracy: {5 * accuracy_factor:.4f}/5')
print('QQP_COMPARISON ' + json.dumps(comparison))


Базовая модель BERT: 0.908335; ваша модель: 0.924388; прирост: +0.016052
Результат базовой модели BERT превышен: True
Коэффициент качества второй части: 1.0000; максимум баллов за вторую часть при этой accuracy: 5.0000/5
QQP_COMPARISON {"baseline_accuracy": 0.908335394509028, "finetuned_accuracy": 0.924387830818699, "delta": 0.016052436309671014, "validation_examples": 40430}


Используйте валидационную выборку для оценки качества и выбора модели. Метки тестовой выборки QQP скрыты (`-1`), поэтому вычислить accuracy на ней локально нельзя.


### Поиск с помощью эмбеддингов BERT (3 балла, обязательная часть)

Теперь попробуем использовать эмбеддинги на основе BERT для поиска информации.

Обычный BERT и похожие модели не обучаются напрямую с контрастивной функцией потерь или функцией потерь для троек примеров, которые сближают эмбеддинги похожих текстов. Поэтому для поисковой системы лучше выбрать специализированную [модель для оценки сходства предложений](https://huggingface.co/models?pipeline_tag=sentence-similarity). Выберите модель, подходящую для этой задачи.

Как и в первом домашнем задании, вам нужно построить поисковую систему:

1. _Подготовьте базу эмбеддингов._ Из пар вопросов QQP возьмите только поле `text1` валидационной выборки (`validation`). Получите эмбеддинги выбранной моделью и сохраните их в `numpy.ndarray` для последующего использования. При желании можно использовать индекс [Faiss](https://github.com/facebookresearch/faiss).
2. _Реализуйте поиск вопросов, похожих на запрос._ Напишите функцию или класс для работы с базой. __Результат этой части оценивается по способности понятно вывести пять наиболее похожих вопросов Quora для произвольного нового запроса.__

Сравните, насколько лучше такой поиск учитывает смысл текстов по сравнению с предыдущим подходом.

Помимо вывода, возвращайте `(row_ids, cosine_scores)` по убыванию косинусной близости. Число результатов равно `min(top_k, len(database))`. Индексы строк базы должны соответствовать порядку исходных `text1`.


In [43]:
# TODO 4: выберите энкодер, создайте database и реализуйте поиск по запросу.
# Инициализируем модель и её токенизатор.
model_name = "sentence-transformers/all-MiniLM-L6-v2" # взял эту модель, она первая на странице по ссылке, наверное есть модели и лучше, но эта в том числе и поменьше по числу параметров и она точно есть в библиотеке
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

all_texts = list(qqp['validation']['text1'])
# Получите по одной строке эмбеддингов NumPy для каждого исходного text1 в том же порядке.
# Используйте этот токенизатор и учитывайте дополнение при усреднении эмбеддингов токенов.
model.to(device)
model.eval()

batch_size = 64
embedding_batches = []

with torch.no_grad():
    for start in tqdm(range(0, len(all_texts), batch_size)):
        texts_batch = all_texts[start:start + batch_size]

        encoded = tokenizer(
            texts_batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        ) # токенизирую
        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        } # делаю словарь, как делали это в начале, для входа модели

        outputs = model(**encoded)
        token_embeddings = outputs.last_hidden_state

        mask = encoded["attention_mask"].unsqueeze(-1)
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1)
        embeddings = summed / counts.clamp(min=1) # clamp для защиты от деления на 0

        embedding_batches.append(embeddings.cpu().numpy())

database = np.concatenate(embedding_batches, axis=0)

def find_similar_questions(query, database, model, top_k = 5):
    """
    Находит и выводит top_k вопросов, наиболее похожих на запрос.

    Кодирует запрос, сравнивает его с заранее вычисленной базой
    эмбеддингов по косинусной близости и выводит наиболее
    похожие по смыслу вопросы.

    Аргументы:
        query (str): Поисковый запрос пользователя.
        database (np.ndarray): Двумерный массив эмбеддингов вопросов в базе.
        model: Модель, используемая со своим токенизатором для кодирования запроса.
        top_k (int): Число результатов для вывода.

    Возвращает:
        tuple: (row_ids, cosine_scores) по убыванию близости, помимо вывода.
    """
    model.eval()
    model_device = next(model.parameters()).device

    encoded = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ) # аналогично токенизирую запрос
    encoded = {
        key: value.to(model_device)
        for key, value in encoded.items()
    } # кладу в виде, нужном модели

    with torch.no_grad():
        outputs = model(**encoded)

    token_embeddings = outputs.last_hidden_state
    mask = encoded["attention_mask"].unsqueeze(-1)

    summed = (token_embeddings * mask).sum(dim=1)
    counts = mask.sum(dim=1)
    query_embedding = summed / counts.clamp(min=1)

    query_vector = query_embedding.cpu().numpy()[0]

    dot_products = database @ query_vector

    query_norm = np.sqrt((query_vector ** 2).sum())
    database_norms = np.sqrt((database ** 2).sum(axis=1))

    scores = dot_products / np.maximum(database_norms * query_norm, 1e-12)

    k = min(top_k, len(database))

    row_ids = np.argsort(scores)[::-1][:k]       # Индексы k максимальных scores, по убыванию
    cosine_scores = scores[row_ids]

    for rank, (row_id, score) in enumerate(
        zip(row_ids, cosine_scores),
        start=1,
    ):
        print(f"{rank}. [{score:.4f}] {all_texts[row_id]}")

    return row_ids, cosine_scores

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9a7b7327-9fd2-4677-94c5-d813ed90db3c)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json
Retrying in 1s [Retry 1/5].
100%|██████████| 632/632 [00:06<00:00, 90.80it/s] 


In [44]:
# Открытые проверки
assert isinstance(database, np.ndarray) and database.ndim == 2
assert database.shape[0] == len(all_texts) == len(qqp['validation'])
assert np.isfinite(database).all() and (np.linalg.norm(database, axis=1) > 0).all()
row_ids, cosine_scores = find_similar_questions("Are apples useful?", database, model, top_k=5)
row_ids, cosine_scores = np.asarray(row_ids), np.asarray(cosine_scores)
assert row_ids.ndim == cosine_scores.ndim == 1
assert len(row_ids) == len(cosine_scores) == min(5, len(database))
assert np.issubdtype(row_ids.dtype, np.integer) and len(set(row_ids.tolist())) == len(row_ids)
assert ((row_ids >= 0) & (row_ids < len(database))).all()
assert np.isfinite(cosine_scores).all() and (np.abs(cosine_scores) <= 1 + 2e-3).all()
assert (np.diff(cosine_scores) <= 1e-6).all(), 'Возвращайте результаты по убыванию косинусной близости.'


1. [0.5971] Is it healthy to eat an apple at night?
2. [0.5638] Can hamsters eat apples?
3. [0.5147] What is the first thing that comes to your mind when you hear the word apple?
4. [0.5147] What is the first thing that comes to your mind when you hear the word apple?
5. [0.5142] What makes Apple the most valuable brand?


In [45]:
find_similar_questions(
    "What are the health benefits of eating apples?",
    database,
    model,
    top_k=5,
)

1. [0.6586] Is it healthy to eat an apple at night?
2. [0.5845] Can hamsters eat apples?
3. [0.5789] What are the health benefits found in the Cupcake Pebbles cereal?
4. [0.5749] What are the health benefits of eating chia seeds?
5. [0.5685] What are the benefits from drinking wine?


(array([11029, 32703, 18242, 13808, 31623]),
 array([0.6586297 , 0.5844608 , 0.5788582 , 0.57491386, 0.56845236],
       dtype=float32))

In [46]:
find_similar_questions(
    "How can I improve my spoken English?",
    database,
    model,
    top_k=5,
)

1. [1.0000] How can I improve my spoken English?
2. [0.9840] How will I improve my spoken English?
3. [0.9417] How can I improve my speaking in english?
4. [0.9407] How do I improve my English speaking?
5. [0.9407] How do I improve my English speaking?


(array([31204,  6412, 16279, 30220, 29372]),
 array([1.0000001 , 0.9839903 , 0.9417277 , 0.94074273, 0.9407426 ],
       dtype=float32))

В предыдущем задании поиск использовал нормализованное среднее FastText-векторов слов. В этом задании представление вопроса получается усреднением выходных представлений токенов модели `sentence-transformers/all-MiniLM-L6-v2` с учётом маски дополнения.

В сохранённых результатах FastText для запроса "How can I find inner peace?" присутствуют релевантные вопросы, например "How can I create inner peace?", но также встречаются вопросы о поиске наставника. Для запроса о будущем искусственного интеллекта в выдачу попадают вопросы о будущем других отраслей. Это показывает, что сходство формулировок может сопровождаться различием темы или намерения.

В текущем поиске для запроса "Are apples useful?" первым найден вопрос "Is it healthy to eat an apple at night?", соответствующий теме пользы яблок. Однако далее встречаются менее релевантные вопросы, включая вопрос о стоимости бренда Apple. Следовательно, полученная выдача также содержит ошибки и не всегда точно отражает намерение запроса.

Использованы разные запросы и разные базы — 537 272 вопроса в предыдущей домашке и 40 430 вопросов в текущей. По этим примерам нельзя достоверно установить превосходство одной модели. Для строгого сравнения необходимо использовать одинаковую базу, одинаковые запросы и общие критерии релевантности.

Результаты FastText тоже приведу:

Query: How can I find inner peace?

Top Matches:
1. How can I create inner peace? (Similarity: 0.9878)
2. How can I find inspiration? (Similarity: 0.9855)
3. How can I find passion? (Similarity: 0.9844)
4. How can I find a career mentor? (Similarity: 0.9837)
5. How can I find a successful mentor? (Similarity: 0.9832)
6. How can I find peace of mind? (Similarity: 0.9829)
7. How can I find a programmer mentor? (Similarity: 0.9820)
8. How can I find app developers? (Similarity: 0.9815)
9. How can I find someone's MSN profile? (Similarity: 0.9812)
10. How can I truly find happiness? (Similarity: 0.9807)

Query: What is the future of artificial intelligence?

Top Matches:
1. What is the future of artificial intelligence? (Similarity: 1.0000)
2. What is the future of Pharmaceutical industry? (Similarity: 0.9968)
3. What is the future of Social Media? (Similarity: 0.9965)
4. What is the future of Chinese economy? (Similarity: 0.9961)
5. What is the nature of public administration? (Similarity: 0.9956)
6. What is the future of Indian badminton? (Similarity: 0.9954)
7. What is the future of social networking? (Similarity: 0.9953)
8. What is the future of India? (Similarity: 0.9950)
9. What is the future of Behavioural Economics? (Similarity: 0.9949)
10. What is the scope of Artificial Intelligence? (Similarity: 0.9948)



P.S. В процессе выполнения ДЗ использовал генеративный ИИ (GPT 6.1 Sol) в качестве тьютора, который мог давать подсказки по коду. Сам код в ноутбуке и выводы написаны полностью мной